In [1]:
import sys
sys.path.append('..')

from Agent.config import LLM, SMALL_LLM, SMART_LLM, EMBEDDING, CLIENT
from IPython.display import display, Image, Markdown

### Market research node

In [2]:
from tabnanny import verbose
from langchain_community.tools.yahoo_finance_news import YahooFinanceNewsTool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent

market_research_agent = create_react_agent(
    SMALL_LLM, 
    tools=[YahooFinanceNewsTool(client=CLIENT), TavilySearchResults(client=CLIENT)],
    state_modifier= '''You are a financial market research analyst supporting external auditors.

Your task:
- Gather recent market news about the specified company and its industry
- Identify events that may impact the company's financial statements 
  (e.g. regulatory changes, lawsuits, M&A, supply chain disruptions)
- Flag any going concern indicators (sharp stock decline, credit rating downgrade, major customer loss)

Output format:
- Company name and ticker
- Key findings (categorized by: industry trends, company-specific events, risk indicators)
- Relevance to financial statement audit (brief assessment)

Do NOT provide investment advice. Focus only on facts relevant to financial statement reliability.'''
)

# state_modifier는 agent의 역할을 정의하는 시스템 프롬프트
# state_modifier를 통해 agent가 시장 조사자 역할을 수행하도록 지시

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
from langgraph.types import  Command
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, AIMessage

def market_research_node(state: MessagesState):
    result = market_research_agent.invoke(state)
    print(f'market_research_node: {result}')
    return Command(
        update= {'messages': [HumanMessage(content=result['messages'][-1].content, name='market_research')]},
        goto= 'supervisor_node'
    )

### DART API 활용 주요정보 받아오는 Node